In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, DateType
from delta.tables import DeltaTable
import uuid
from datetime import datetime

PIPELINE     = "people"
LAYER        = "silver"
BRONZE_TABLE = "workspace.bronze.bronze_people"
SILVER_TABLE = "workspace.silver.silver_people"
AUDIT_TABLE  = "workspace.audit.people_pipeline_runs"
MERGE_KEY    = "user_id"


In [0]:
run_id   = str(uuid.uuid4())
start_ts = datetime.utcnow()

wm_df = spark.sql(f"""
    SELECT MAX(watermark_out) AS last_wm
    FROM {AUDIT_TABLE}
    WHERE pipeline = '{PIPELINE}'
      AND layer    = '{LAYER}'
      AND status   = 'SUCCESS'
""")

last_watermark = wm_df.collect()[0]["last_wm"]
if last_watermark is None:
    last_watermark_val = "1900-01-01 00:00:00"
else:
    last_watermark_val = str(last_watermark)

print(f"Watermark used: {last_watermark_val}")

df_bronze = spark.read.table(BRONZE_TABLE)
df_new    = df_bronze.where(
    F.col("ingest_ts") > F.lit(last_watermark_val).cast("timestamp")
)

rows_read = df_new.count()
print(f"New rows to process: {rows_read}")


In [0]:
if rows_read > 0:
    df_silver = df_new \
        .drop("index") \
        .withColumn("user_id",       F.col("user_id").cast(StringType())) \
        .withColumn("first_name",    F.col("first_name").cast(StringType())) \
        .withColumn("last_name",     F.col("last_name").cast(StringType())) \
        .withColumn("sex",           F.col("sex").cast(StringType())) \
        .withColumn("email",         F.col("email").cast(StringType())) \
        .withColumn("phone",         F.col("phone").cast(StringType())) \
        .withColumn("job_title",     F.col("job_title").cast(StringType())) \
        .withColumn("date_of_birth", F.col("date_of_birth").cast(DateType())) \
        .dropDuplicates(["user_id"])

    if not spark.catalog.tableExists(SILVER_TABLE):
        df_silver.write.format("delta").mode("overwrite").saveAsTable(SILVER_TABLE)
        rows_upserted = df_silver.count()
        print(f"Silver created fresh. Rows written: {rows_upserted}")
    else:
        target = DeltaTable.forName(spark, SILVER_TABLE)
        target.alias("tgt").merge(
            df_silver.alias("src"),
            f"tgt.{MERGE_KEY} = src.{MERGE_KEY}"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
        rows_upserted = rows_read
        print(f"Silver merged. Rows upserted: {rows_upserted}")
else:
    rows_upserted = 0
    print("No new rows. Silver unchanged. Idempotent run ✅")

end_ts        = datetime.utcnow()
new_watermark = df_bronze.agg(F.max("ingest_ts")).collect()[0][0]

from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType
from datetime import datetime as dt

wm_in_ts = dt(1900, 1, 1, 0, 0, 0) if last_watermark is None else last_watermark

audit_schema = StructType([
    StructField("run_id",               StringType(),    True),
    StructField("pipeline",             StringType(),    True),
    StructField("layer",                StringType(),    True),
    StructField("start_ts",             TimestampType(), True),
    StructField("end_ts",               TimestampType(), True),
    StructField("status",               StringType(),    True),
    StructField("watermark_in",         TimestampType(), True),
    StructField("watermark_out",        TimestampType(), True),
    StructField("rows_read_bronze",     LongType(),      True),
    StructField("rows_upserted_silver", LongType(),      True),
    StructField("error_message",        StringType(),    True),
])

audit_row = spark.createDataFrame([{
    "run_id":               run_id,
    "pipeline":             PIPELINE,
    "layer":                LAYER,
    "start_ts":             start_ts,
    "end_ts":               end_ts,
    "status":               "SUCCESS",
    "watermark_in":         wm_in_ts,
    "watermark_out":        new_watermark,
    "rows_read_bronze":     rows_read,
    "rows_upserted_silver": rows_upserted,
    "error_message":        "none"
}], schema=audit_schema)

audit_row.write.format("delta").mode("append").saveAsTable(AUDIT_TABLE)

print(f"\n✅ Done | run_id: {run_id}")
print(f"   rows_read_bronze:     {rows_read}")
print(f"   rows_upserted_silver: {rows_upserted}")
print(f"   watermark_out:        {new_watermark}")


In [0]:
# Verify Silver
df_s = spark.read.table("workspace.silver.silver_people")
print("Silver row count:", df_s.count())
print("Silver columns:", df_s.columns)
df_s.show(3)


In [0]:
# Verify Audit
spark.read.table("workspace.audit.people_pipeline_runs") \
    .select("run_id","status","watermark_in","watermark_out",
            "rows_read_bronze","rows_upserted_silver") \
    .show(truncate=False)


In [0]:
spark.read.table("workspace.audit.people_pipeline_runs").printSchema()


In [0]:
spark.read.table("workspace.audit.people_pipeline_runs") \
    .select("run_id", "status", "watermark_in", "watermark_out",
            "rows_read_bronze", "rows_upserted_silver") \
    .show(truncate=False)


In [0]:
from pyspark.sql import functions as F
import time

time.sleep(2)  # ensure new ingest_ts is strictly greater than watermark

df_existing = spark.read.table("workspace.bronze.bronze_people")

df_new_batch = df_existing.limit(5) \
    .withColumn("user_id",   F.concat(F.lit("NEW_"), F.col("user_id"))) \
    .withColumn("ingest_ts", F.current_timestamp())

df_new_batch.write.format("delta").mode("append").saveAsTable("workspace.bronze.bronze_people")

print("5 new rows added to Bronze")
spark.read.table("workspace.bronze.bronze_people").agg(F.count("*"), F.max("ingest_ts")).show()
